# 02 — Baseline forecasting

Mục tiêu: tạo các mô hình đối chứng đơn giản cho **Rossmann Store 1**, trước khi đánh giá LSTM. Cửa sổ lịch sử là 30 ngày; baseline dùng lịch sử quá khứ, không dùng ngẫu nhiên hóa dữ liệu.

**Các baseline:** Naive (ngày trước), Seasonal Naive (cùng thứ tuần trước), Moving Average 7 ngày và 30 ngày. Đánh giá trên validation và holdout theo thời gian. Không dùng `official_test_df` vì tập test Kaggle không có nhãn `Sales`.

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
RESULT_DIR = ARTIFACT_DIR / "results"
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
for folder in (RESULT_DIR, CHECKPOINT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

LOOKBACK = 30
STORE_ID = 1

from src.data import load_rossmann_data, chronological_split
from src.features import (
    create_calendar_features, prepare_features, get_feature_columns,
    fit_feature_scaler, transform_features, fit_target_scaler,
    transform_target, inverse_transform_target, make_sequences,
    DEFAULT_LOOKBACK
)

train_df, official_test_df = load_rossmann_data(DATA_DIR, store_id=STORE_ID)
train_df = train_df.sort_values("Date").reset_index(drop=True)
splits = chronological_split(train_df, train_ratio=0.70, val_ratio=0.15, holdout_ratio=0.15)
print(f"Store: {STORE_ID} | labeled rows: {len(train_df):,}")
print(f"Train:      {splits['train']['Date'].min().date()} -> {splits['train']['Date'].max().date()} ({len(splits['train'])})")
print(f"Validation: {splits['validation']['Date'].min().date()} -> {splits['validation']['Date'].max().date()} ({len(splits['validation'])})")
print(f"Holdout:    {splits['holdout']['Date'].min().date()} -> {splits['holdout']['Date'].max().date()} ({len(splits['holdout'])})")

## 1. Định nghĩa metric và baseline

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "WAPE_pct": np.nan}
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    denom = np.sum(np.abs(y_true))
    wape = 100 * np.sum(np.abs(y_true - y_pred)) / denom if denom else np.nan
    return {"MAE": mae, "RMSE": rmse, "WAPE_pct": wape}

def baseline_predictions(df):
    """Forecast target at t using sales strictly before t."""
    out = df[["Date", "Store", "Sales", "Open", "Promo"]].copy()
    sales = df["Sales"].astype(float)
    out["Naive_1d"] = sales.shift(1)
    out["SeasonalNaive_7d"] = sales.shift(7)
    out["MovingAverage_7d"] = sales.shift(1).rolling(7, min_periods=7).mean()
    out["MovingAverage_30d"] = sales.shift(1).rolling(30, min_periods=30).mean()
    return out

baseline_df = baseline_predictions(train_df)
baseline_df.tail()

## 2. Đánh giá validation và holdout

In [ ]:
rows = []
for split_name in ("validation", "holdout"):
    start_date = splits[split_name]["Date"].min()
    end_date = splits[split_name]["Date"].max()
    part = baseline_df.loc[baseline_df["Date"].between(start_date, end_date)].copy()
    for model_name in ("Naive_1d", "SeasonalNaive_7d", "MovingAverage_7d", "MovingAverage_30d"):
        metrics = regression_metrics(part["Sales"], part[model_name])
        rows.append({"split": split_name, "model": model_name, **metrics})

baseline_metrics = pd.DataFrame(rows)
display(baseline_metrics.sort_values(["split", "WAPE_pct"]))

## 3. Biểu đồ holdout

In [ ]:
holdout_start = splits["holdout"]["Date"].min()
holdout_end = splits["holdout"]["Date"].max()
plot_df = baseline_df.loc[baseline_df["Date"].between(holdout_start, holdout_end)].copy()

plt.figure(figsize=(14, 5))
plt.plot(plot_df["Date"], plot_df["Sales"], label="Actual", linewidth=2)
plt.plot(plot_df["Date"], plot_df["SeasonalNaive_7d"], label="Seasonal Naive (7d)", alpha=.85)
plt.plot(plot_df["Date"], plot_df["MovingAverage_30d"], label="Moving Average (30d)", alpha=.85)
plt.title("Rossmann Store 1 — Baseline forecasts on holdout")
plt.xlabel("Date"); plt.ylabel("Sales"); plt.legend(); plt.tight_layout()
plt.show()

## 4. Lưu kết quả để các notebook sau sử dụng

In [ ]:
baseline_out = RESULT_DIR / "baseline_holdout_predictions.csv"
metrics_out = RESULT_DIR / "baseline_metrics.csv"
plot_df.to_csv(baseline_out, index=False)
baseline_metrics.to_csv(metrics_out, index=False)
print("Saved:", baseline_out)
print("Saved:", metrics_out)

### Diễn giải
- Seasonal Naive thường là baseline đáng chú ý vì doanh số có cấu trúc theo thứ trong tuần.
- MAE/RMSE càng thấp càng tốt; WAPE càng thấp càng tốt.
- Dùng validation để lựa chọn baseline. Holdout nên được giữ cho đánh giá cuối cùng và không dùng để tinh chỉnh tham số.